<a href="https://colab.research.google.com/github/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model/blob/dev/SD15_NPU_Official_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SD 1.5 → Qualcomm NPU (Local Dream / Ruya) — RESMİ HAT

Bu defter, Local Dream'in **kendi dönüştürme scriptlerini** (`npuconvertv2`)
QNN SDK **2.28** ile koşar. Rehber: `ld-guide.chino.icu/conversion/sd15`

**Neden bu hat:** kendi yazdığımız hat cihazda yüklenmeyen paketler üretiyordu.
Sebepleri resmi scriptlerde görüldü:

| bizim (eski) | resmi |
|---|---|
| `qairt-converter` → DLC | `qnn-onnx-converter` → `model.cpp` → `.so` |
| `--act_bitwidth 8` + io16 hilesi | **`--act_bitwidth 16`** |
| per-channel kapalı | `--use_per_channel_quantization` |
| stok diffusers | `redefined_modules/` (MHA→SHA, Linear→Conv) |
| VTCM ayarsız | `"vtcm_mb": 2` |

**Çalışma sırası:** 1 → 2 → 3 → 4 → 5

> **Çalışma zamanı seçimi (GPU mu, RAM mi?):** GPU **yalnızca** kalibrasyon
> verisi üretimini (`prepare_data.py`) hızlandırır — çözünürlük başına ~35 dk
> yerine ~3 dk. Kuantizasyon, model-lib ve context-binary adımları **tamamen
> CPU**'dur ve GPU'dan hiç etkilenmez; onların tek kısıtı **RAM**'dir (~20 GB).
> Ücretsiz Colab'da GPU seçmek RAM'i 51 GB'tan ~13 GB'a düşürür ve kuantizasyon
> OOM olur — o yüzden **ücretsiz hesapta yüksek bellekli CPU** çalışma zamanı
> seçin. Colab Pro'da L4/A100 hem GPU hem ~51 GB verir; ikisi birden isteniyorsa
> tek doğru seçim odur.
>
> Torch'un gerçekten CUDA gördüğü 5. adımın başında `[cuda] torch ... CUDA
> ETKIN` satırıyla **ölçülerek** yazılır. `GPU VAR ama torch ... CUDA
> goremiyor` diyorsa `prepare_data` CPU'da koşuyor demektir.

> **Çözünürlük notu:** NPU'da çözünürlük istek parametresi değil **derleme
> zamanı** özelliğidir. Taban `unet.bin` 512×512 içindir; diğer boyutlar
> pakete birer **zstd yaması** (`768.patch`, `512x768.patch` …) olarak girer ve
> uygulama açılışta bunu uygular. 1. adımdaki kutulardan seçin — her ek boyut
> **tam bir dönüştürme turu** demektir (kalibrasyon + kuantizasyon baştan).


## 1) Ayarlar

In [ ]:
#@title Ayarlar { display-mode: "form" }
#@markdown Model dosyasının **doğrudan indirme** bağlantısı (.safetensors)
SAFETENSORS_URL = "https://civitai.com/api/download/models/2681234?fileId=2567874"  #@param {type:"string"}
#@markdown Model adı — **sade** yazın. Sürüm ve SOC eki (`_qnn2.28_8gen2`)
#@markdown paket adına zaten eklenir.
MODEL_NAME = "CyberRealistic"  #@param {type:"string"}
#@markdown Çip katmanı — `min` = Hexagon V68+ (Snapdragon 7 Gen 1 dahil)
SOC = "min"  #@param ["min", "8gen1", "8gen2"]
#@markdown `clip_skip`: **SD1.5'in varsayılanı 1'dir.** Anime/NovelAI türevleri
#@markdown genelde 2 ister. Model kartında bir değer YAZMIYORSA 1 bırakın —
#@markdown "yazmıyorsa 2'dir" yaygın ama yanlış bir varsayım. Bu değer pakete
#@markdown gömülür, telefonda değiştirilemez.
CLIP_SKIP = 1  #@param [1, 2] {type:"raw"}
#@markdown Foto-gerçekçi model ise işaretleyin: kalibrasyon promptları anime
#@markdown yerine fotoğraf sahnelerine döner.
REALISTIC = True  #@param {type:"boolean"}
#@markdown Kuantizasyon örnek sayısı (UNet). `0` = resmi tarifin tamamı (400) —
#@markdown **üretim için bunu kullanın**. `24` yalnızca boru hattını doğrulamak
#@markdown içindir, kalite düşer. İki modeli karşılaştıracaksanız ikisinde de
#@markdown AYNI değeri kullanın, yoksa fark modelin değil ayarın farkı olur.
CALIB_LIMIT = 0  #@param {type:"integer"}
#@markdown GPU varsa CUDA torch kur (prepare_data ~10x hızlanır)
CUDA_TORCH = True  #@param {type:"boolean"}

#@markdown ---
#@markdown ### Ek çözünürlükler
#@markdown **512×512 her zaman üretilir** — yamaların tabanı odur, seçime gerek yok.
#@markdown İşaretlediğiniz her boyut için kalibrasyon + kuantizasyon **baştan**
#@markdown koşar; süre kabaca taban koşu × (1 + seçilen boyut sayısı) olur.
#@markdown Çıktı, paketin içine `768.patch` / `512x768.patch` olarak girer ve
#@markdown uygulamanın çözünürlük listesinde görünür.
RES_512x768 = False  #@param {type:"boolean"}
RES_768x512 = False  #@param {type:"boolean"}
RES_768x768 = False  #@param {type:"boolean"}
RES_768x1024 = False  #@param {type:"boolean"}
RES_1024x768 = False  #@param {type:"boolean"}
RES_1024x1024 = False  #@param {type:"boolean"}

import os
RESOLUTIONS = ",".join(name for name, secili in (
    ("512x768", RES_512x768),
    ("768x512", RES_768x512),
    ("768x768", RES_768x768),
    ("768x1024", RES_768x1024),
    ("1024x768", RES_1024x768),
    ("1024x1024", RES_1024x1024),
) if secili)

os.environ.update(
    MODEL_NAME=MODEL_NAME, SOC=SOC,
    CLIP_SKIP=str(CLIP_SKIP),
    REALISTIC="1" if REALISTIC else "0",
    CALIB_LIMIT=str(CALIB_LIMIT),
    CUDA_TORCH="1" if CUDA_TORCH else "0",
    SAFETENSORS_URL=SAFETENSORS_URL,
    RESOLUTIONS=RESOLUTIONS,
)
print(f"{MODEL_NAME} | soc={SOC} clip_skip={CLIP_SKIP} realistic={REALISTIC}")
print(f"calib_limit={CALIB_LIMIT} cuda_torch={CUDA_TORCH}")
if CALIB_LIMIT and CALIB_LIMIT < 150:
    print(f"[!] CALIB_LIMIT={CALIB_LIMIT}: bu bir TEST degeri. Uretim paketi icin 0\n"
          "    (resmi 400) kullanin; dusuk deger plastik ten / bozulma uretebilir.")
print("cozunurluk: 512x512 (taban)" + (f" + {RESOLUTIONS}" if RESOLUTIONS else ""))
if RESOLUTIONS:
    if SOC == "min":
        print("[!] SOC=min: resmi tarif dusuk HTP kusaklari icin ek cozunurluk\n"
              "    uretmiyor (v68 yuksek cozunurlugu kaldiramiyor). Yama uretilir\n"
              "    ama cihazda yuklenmeyebilir — 8gen1/8gen2 onerilir.")
    if "1024" in RESOLUTIONS:
        print("[!] 1024 kenar: kuantizasyon RAM'i 512'ye gore ~4x. Once 768 ile\n"
              "    dogrulayin; ayrica cihazda VTCM'ye sigmayabilir.")
!nvidia-smi -L || echo "GPU yok — prepare_data yavas olacak"


## 2) Depo + araçlar

Depo `dev` dalından çekilir. Ayar değiştirdiyseniz bu hücreyi tekrar koşmanız
yeterli — not defterini yeniden açmaya gerek yok.


In [ ]:
%cd /content
REPO = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model"
BRANCH = "dev"
import os, subprocess
if not os.path.isdir("/content/sd-qnn/.git"):
    !git clone -b {BRANCH} {REPO} /content/sd-qnn
else:
    !cd /content/sd-qnn && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/sd-qnn
!pip install -q uv
!git log --oneline -1


## 3) QNN SDK 2.28

~2 GB. **Sürüm önemli** — rehber 2.28 şart koşuyor. İndirme koparsa bu hücreyi
tekrar çalıştırın, kaldığı yerden devam eder.

In [ ]:
%cd /content/sd-qnn
import os
URL = ("https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/"
       "qualcomm_neural_processing_sdk/v2.28.0.241029.zip")
out = !python3 scripts/setup_qnn_sdk.py --dest /content/qairt --asset-url "{URL}"
print("\n".join(out[-25:]))
root = [l.split("=",1)[1] for l in out if l.startswith("QNN_SDK_ROOT=")]
assert root, "QNN_SDK_ROOT bulunamadi — yukaridaki ciktiya bakin"
os.environ["QNN_SDK_ROOT"] = root[-1].strip()
print("\nQNN_SDK_ROOT =", os.environ["QNN_SDK_ROOT"])


## 4) Modeli indir

Resmi hat `.safetensors` dosyasını **doğrudan** kullanır. İndirme mantığı
`scripts/fetch_ckpt.py` içinde: token ekleme, yarım dosyadan devam, HTML/eksik
dosya kontrolü.

> **civitai** indirme uçları artık **token** ister (tokensiz `401`).
> civitai.com → **Account settings → API Keys → Add API key**, sonra Colab
> **🔑 Secrets → `CIVITAI_TOKEN`** (*Notebook access* açık). Gated **HF** linki
> için aynı şekilde `HF_TOKEN`.


In [ ]:
%cd /content/sd-qnn
import os
try:
    from google.colab import userdata
    for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
        v = userdata.get(k)
        if v: os.environ[k] = v
except Exception as e:
    print("[!] Secrets:", e)
!python scripts/fetch_ckpt.py --out work/input.safetensors


## 5) Dönüştür

Aşamalar: `uv` ortamı → `prepare_data` → `gen_quant_data` → `export_onnx` →
`qnn-onnx-converter` → `qnn-model-lib-generator` → `qnn-context-binary-generator`

Seçilen her ek çözünürlük için bu tur **UNet'e özel** olarak tekrarlanır
(`export_onnx_unet_only` → `convert_all_unet_only`) ve çıkan `unet.bin`,
taban 512×512 binary'sine karşı `zstd --patch-from` ile farklanıp pakete
`768.patch` / `512x768.patch` olarak konur.

> **Sekmeyi ön planda tutun.** Mobilde başka uygulamaya geçince tarayıcı sekmeyi
> askıya alıyor ve Colab çalışma zamanı kapanıyor. `CACHE_REPO` doluysa hem
> `prepare_data` çıktısı (her çözünürlük için ayrı) hem de o ana kadar üretilmiş
> **binary + yamalar** HF'e yedeklenir; kopan oturum kaldığı yerden devam eder.
> Hücre tekrar çalıştırıldığında biten çözünürlükler **atlanır**.


In [ ]:
#@title Dönüştür { display-mode: "form" }
#@markdown **Onbellek deposu** (HF, ozel dataset): pahali asamalar buraya
#@markdown yedeklenir — her cozunurlugun `data.pkl` + `images/` ciktisi ve
#@markdown birikmis `output_512/` (taban binary + yamalar). Calisma zamani
#@markdown kapanirsa yeni oturum indirip kaldigi yerden devam eder. Bos = kapali.
CACHE_REPO = "sd-qnn-cache"  #@param {type:"string"}
#@markdown Ciktiyi da yedekle (binary + yamalar, ~1.5 GB). Kapatirsaniz yalnizca
#@markdown kalibrasyon verisi yedeklenir; kopan oturum kuantizasyonu bastan yapar.
CACHE_OUTPUT = True  #@param {type:"boolean"}

%cd /content/sd-qnn
import os
assert os.environ.get("QNN_SDK_ROOT"), "Once 3. hucreyi calistirin"
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t: os.environ["HF_TOKEN"] = t
except Exception as e:
    print("[!] Secrets:", e)
os.environ["CACHE_REPO"] = CACHE_REPO.strip()
os.environ["CACHE_OUTPUT"] = "1" if CACHE_OUTPUT else "0"
if CACHE_REPO.strip() and not os.environ.get("HF_TOKEN"):
    print("[!] Onbellek icin HF_TOKEN gerekli (Secrets) — onbellek kapali")
name = os.environ["MODEL_NAME"]
print("cozunurluk: 512x512 (taban)" +
      (f" + {os.environ['RESOLUTIONS']}" if os.environ.get("RESOLUTIONS") else ""))
# Calisma dizini adinda BOSLUK olmamali: resmi convert_all.sh `cd ${current_pwd}`
# satirini tirnaksiz yaziyor ve bosluklu yolda "cd: too many arguments" ile
# duruyor. Model adi ("epiCRealism Natural Sin") oldugu gibi kalir, yalnizca
# dizin adi sadelesir; onceki oturumun isi varsa tasinir, bastan kosulmaz.
import re, shutil
slug = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("_") or "model"
work = f"work/{slug}"
if slug != name and os.path.isdir(f"work/{name}") and not os.path.isdir(work):
    shutil.move(f"work/{name}", work)
    print(f"[yol] 'work/{name}' -> '{work}' (bosluksuz)")
!bash scripts/06_official_pipeline.sh work/input.safetensors "{name}" "{work}" "$SOC"


## 6) Paketi indir

ZIP'i telefona kopyalayıp **Ruya / Local Dream → Ayarlar → Modeli içe aktar**
ile ekleyin. Paketteki `*.patch` dosyaları uygulamanın çözünürlük listesini
belirler: yaması olmayan bir boyut seçilemez.


In [ ]:
import glob, os, re, zipfile
from google.colab import files
zips = sorted(glob.glob("/content/sd-qnn/dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ bos — donusum tamamlanmadi"
z = zips[-1]
print(f"{z}  ({os.path.getsize(z)/1e6:.0f} MB)")

# Paketin gercekten hangi cozunurlukleri destekledigini yamalardan oku —
# uygulama da tam olarak boyle tariyor.
kare, dikdortgen = re.compile(r"^(\d+)\.patch$"), re.compile(r"^(\d+)x(\d+)\.patch$")
boyutlar = {(512, 512)}
with zipfile.ZipFile(z) as zf:
    for n in zf.namelist():
        b = os.path.basename(n)
        m = kare.match(b)
        if m: boyutlar.add((int(m.group(1)), int(m.group(1))))
        m = dikdortgen.match(b)
        if m: boyutlar.add((int(m.group(1)), int(m.group(2))))
print("cozunurlukler:", ", ".join(f"{w}x{h}" for w, h in sorted(boyutlar, key=lambda t: t[0]*t[1])))
!unzip -l "{z}"
files.download(z)


## 7) Hugging Face'e yükle (isteğe bağlı)

Telefona indirmek yerine (ya da ek olarak) ZIP'i HF'e koyar: çalışma zamanı
kapansa bile kalıcı olur ve telefondan doğrudan indirilebilir.

**Gerekli:** yazma (write) izinli token → Colab sol menü **🔑 Secrets →
`HF_TOKEN`** (*Notebook access* açık olmalı).

Alanlar bu hücrenin kendi formunda — başka hücreye bağımlı değil.


In [ ]:
#@title Hugging Face'e yükle { display-mode: "form" }
#@markdown **Ayri repo:** her model `<kullanici>/<MODEL_NAME>` reposuna gider.
#@markdown **Koleksiyon:** hepsi tek repoda, her model kendi alt klasorunde.
UPLOAD_MODE = "Ayri repo"  #@param ["Ayri repo", "Koleksiyon"]
#@markdown Koleksiyon modunda kullanilacak repo adi
COLLECTION_REPO = "sd_qnn"  #@param {type:"string"}
#@markdown Elle tam repo adi (`kullanici/repo`) — doluysa yukaridakiler yok sayilir
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}

import glob, os, subprocess, sys
os.chdir("/content/sd-qnn")
assert os.path.isdir("scripts"), "Once 2. adimi (depoyu cek) calistirin!"

zips = sorted(glob.glob("dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ icinde zip yok — 5. adim (donusum) tamamlanmamis olabilir."
zip_path = zips[-1]
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.0f} MB)")

# Token: Colab Secrets -> HF_TOKEN (write izinli olmali)
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except Exception as e:
    print("[!] Secrets okunamadi:", e)
assert os.environ.get("HF_TOKEN"), \
    "HF_TOKEN yok — Colab Secrets (anahtar simgesi) -> HF_TOKEN ekleyin (write)."

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub"], check=True)

# 1. hucre calistirilmadiysa model adini zip isminden turet
name = os.environ.get("MODEL_NAME", "").strip() or \
    os.path.basename(zip_path).split("_qnn")[0]

cmd = [sys.executable, "scripts/upload_hf.py", "--file", zip_path,
       "--name", name]
if HF_REPO.strip():
    cmd += ["--repo", HF_REPO.strip()]
elif UPLOAD_MODE == "Koleksiyon":
    cmd += ["--collection", COLLECTION_REPO.strip()]
if HF_PRIVATE:
    cmd.append("--private")

print(">", " ".join(cmd))
rc = subprocess.run(cmd).returncode
assert rc == 0, f"Yukleme basarisiz (cikis kodu {rc}) — yukaridaki hataya bakin."
